# Projet MLSF – Partie I : solveur SAT pour Meowdoku
**Pablo Bosch, Cécile Chevrière**

Les programmes sont regroupés dans `meowdoku.py` (grilles, encodage CNF, DIMACS, vérification) et `generateur.py` (génération de grilles). Ce notebook les exécute dans l'ordre des questions ; la démarche et l'analyse des résultats sont dans le rapport.

In [1]:
import itertools, os, random
from meowdoku import (lire_grille, ecrire_grille, afficher, grille_vers_cnf, ecrire_dimacs,
                      convertir, var, satisfait, chats_depuis_valuation, verifier_solution)
from generateur import generer_grille, generer_fichier

os.makedirs("grilles", exist_ok=True)
os.makedirs("cnf", exist_ok=True)

## Question 1 – Modélisation de Meowdoku

### 1.1 Conversion grille → DIMACS
Grille de la figure 1 de l'énoncé : lecture, conversion, aperçu du fichier produit.

In [2]:
grille = lire_grille("grilles/grille_enonce.txt")
afficher(grille)
nb_vars, nb_clauses = convertir("grilles/grille_enonce.txt", "cnf/grille_enonce.cnf")
print(f"\n{nb_vars} variables, {nb_clauses} clauses")
with open("cnf/grille_enonce.cnf") as f:
    print("".join(f.readlines()[:10]) + "...")

A B C B B D
A B B B B D
A A B D D D
A A A E D D
A A A A D D
F A D D D D

36 variables, 309 clauses
c Meowdoku : grilles/grille_enonce.txt
c GRID 6 6, 6 regions
c variable x_(i,j) = (i-1)*6 + j  (chat sur la case (i,j))
p cnf 36 309
-1 -2 0
-1 -3 0
-1 -4 0
-1 -5 0
-1 -6 0
-2 -3 0
...


### 1.2 Validation de l'encodage
**a.** La solution de la figure 2 est licite et satisfait la CNF.

In [3]:
solution_fig2 = {(3, 2), (2, 5), (1, 3), (5, 6), (4, 4), (6, 1)}
_, clauses = grille_vers_cnf(grille)
print("règles violées :", verifier_solution(grille, solution_fig2))
print("CNF satisfaite :", satisfait(clauses, {var(i, j, grille.M) for i, j in solution_fig2}))

règles violées : []
CNF satisfaite : True


**b.** Test exhaustif sur des grilles 4×4 : pour chacune des 2¹⁶ valuations, on compare « la valuation satisfait la CNF » et « la configuration de chats associée est licite ».

In [4]:
def comparer(grille, clauses, vraies):
    return satisfait(clauses, vraies) == (not verifier_solution(grille, chats_depuis_valuation(vraies, grille.M)))

for graine in range(5):
    g, _ = generer_grille(4, graine)
    nv, cl = grille_vers_cnf(g)
    modeles, accord = 0, True
    for bits in range(2 ** nv):
        vraies = {v for v in range(1, nv + 1) if bits >> (v - 1) & 1}
        accord &= comparer(g, cl, vraies)
        modeles += satisfait(cl, vraies)
    print(f"grille 4x4 n°{graine} : accord sur les 2^{nv} valuations = {accord}, modèles = {modeles}")

grille 4x4 n°0 : accord sur les 2^16 valuations = True, modèles = 1
grille 4x4 n°1 : accord sur les 2^16 valuations = True, modèles = 1
grille 4x4 n°2 : accord sur les 2^16 valuations = True, modèles = 2
grille 4x4 n°3 : accord sur les 2^16 valuations = True, modèles = 1
grille 4x4 n°4 : accord sur les 2^16 valuations = True, modèles = 2


**c.** Grilles plus grandes (2^NM valuations hors de portée) : toutes les permutations (un chat par ligne et par colonne) et 20 000 valuations aléatoires.

In [5]:
rng = random.Random(0)
for N, graine in [(6, 0), (7, 1)]:
    g, _ = generer_grille(N, graine)
    nv, cl = grille_vers_cnf(g)
    perms = [{var(i + 1, p[i] + 1, N) for i in range(N)} for p in itertools.permutations(range(N))]
    aleatoires = [{v for v in range(1, nv + 1) if rng.random() < rng.choice([0.05, 0.15, 0.3])}
                  for _ in range(20000)]
    ok = all(comparer(g, cl, s) for s in perms + aleatoires)
    print(f"{N}x{N} : {len(perms)} permutations + 20000 valuations aléatoires, accord = {ok}, "
          f"permutations solutions = {sum(satisfait(cl, s) for s in perms)}")

6x6 : 720 permutations + 20000 valuations aléatoires, accord = True, permutations solutions = 19
7x7 : 5040 permutations + 20000 valuations aléatoires, accord = True, permutations solutions = 166


### 1.3 Générateur de grilles (N = M = K)
Exemple de grille 8×8 générée, avec la solution construite (`*`).

In [6]:
g, chats = generer_fichier(8, 2026, "grilles/exemple_8x8.txt")
afficher(g, chats)
print("\nrègles violées par la solution construite :", verifier_solution(g, chats))

B B B E A A * A
* B E E A A A C
B B E E A C C *
B E E E * C C C
E E * D D D F F
G G G H F * F F
G * G H H F F F
G G H * H H F F

règles violées par la solution construite : []


Pas de configuration licite pour N = 2 et N = 3 :

In [7]:
for N in (2, 3):
    try:
        generer_grille(N)
    except ValueError as e:
        print(e)

aucune configuration licite pour N = 2 (impossible pour N = 2, 3)
aucune configuration licite pour N = 3 (impossible pour N = 2, 3)


### 1.4 Jeu de grilles générées et fichiers DIMACS
Grilles de taille 4 à 20 (3 graines par taille), vérification de la solution construite et conversion en DIMACS. Ces instances serviront aux questions suivantes.

In [8]:
print(f"{'N':>3} {'variables':>10} {'clauses':>9}  solutions construites licites")
for N in range(4, 21):
    tailles, licites = [], True
    for graine in range(3):
        nom = f"meow_{N:02d}_{graine}"
        g, chats = generer_fichier(N, graine, f"grilles/{nom}.txt")
        licites &= not verifier_solution(g, chats)
        tailles.append(convertir(f"grilles/{nom}.txt", f"cnf/{nom}.cnf"))
    nv = tailles[0][0]
    nc = sum(t[1] for t in tailles) // 3
    print(f"{N:>3} {nv:>10} {nc:>9}  {licites}")

  N  variables   clauses  solutions construites licites
  4         16        74  True
  5         25       146  True
  6         36       258  True
  7         49       412  True
  8         64       638  True
  9         81       909  True
 10        100      1253  True
 11        121      1671  True
 12        144      2207  True
 13        169      2807  True
 14        196      3616  True
 15        225      4341  True
 16        256      5423  True
 17        289      6568  True
 18        324      7720  True
 19        361      9039  True
 20        400     10547  True


## Question 2 – Solveur DPLL

La lecture DIMACS (`lire_dimacs`) et les algorithmes 1 et 2 (`dpll_statut`, `dpll`) sont dans `dpll.py`.

In [ ]:
from dpll import lire_dimacs, dpll_statut, dpll, completer, verifier_modele

### Validation
**a.** Comparaison avec la force brute ($2^n$ valuations) sur 500 petites formules aléatoires.

In [11]:
rng = random.Random(0)
nb_sat = 0
for _ in range(500):
    n = rng.randint(1, 8)
    F = [[rng.choice((1, -1)) * rng.randint(1, n) for _ in range(rng.randint(1, 3))]
         for _ in range(rng.randint(0, 30))]
    ref = any(verifier_modele(F, [i + 1 if b else -(i + 1) for i, b in enumerate(bits)])
              for bits in itertools.product((0, 1), repeat=n))
    assert dpll_statut(F, rng) == ref
    sat, v = dpll(F, rng)
    assert sat == ref and (not sat or verifier_modele(F, completer(v, n)))
    nb_sat += sat
print(f"500 formules : accord avec la force brute ({nb_sat} SAT, {500 - nb_sat} UNSAT)")

500 formules : accord avec la force brute (154 SAT, 346 UNSAT)


**b.** Instances fournies par les enseignants (`../../public-instances/`), avec la limite de 60 s de la compétition. Chaque exécution passe par la ligne de commande `python dpll.py` dans un processus séparé, ce qui permet de l'interrompre ; les modèles renvoyés sont vérifiés.

In [12]:
import json, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

LIMITE = 60
attendu = json.load(open("../../public-instances/expected.json"))

def executer(nom, algo, graine=0):
    chemin = f"../../public-instances/instances/{nom}"
    t0 = time.perf_counter()
    try:
        r = subprocess.run([sys.executable, "dpll.py", chemin, "--algo", str(algo),
                            "--graine", str(graine)], capture_output=True, text=True, timeout=LIMITE)
    except subprocess.TimeoutExpired:
        return "TIMEOUT", None, LIMITE
    duree = time.perf_counter() - t0
    sortie = r.stdout.splitlines()
    statut = "SAT" if "s SATISFIABLE" in sortie else "UNSAT"
    correct = statut == attendu[nom]
    if correct and statut == "SAT" and algo == 2:
        modele = [int(x) for x in sortie[1].split()[1:-1]]
        correct = verifier_modele(lire_dimacs(chemin)[1], modele)
    return statut, correct, duree

taches = [(nom, algo) for nom in sorted(attendu) for algo in (1, 2)]
with ThreadPoolExecutor(4) as ex:
    resultats = dict(zip(taches, ex.map(lambda t: executer(*t), taches)))

print(f"{'instance':<34}{'attendu':>8}   {'algo 1':>16}   {'algo 2':>16}")
for nom in sorted(attendu):
    cases = []
    for algo in (1, 2):
        statut, correct, duree = resultats[nom, algo]
        cases.append("timeout" if statut == "TIMEOUT"
                     else f"{statut} {'ok' if correct else 'FAUX'} {duree:5.2f}s")
    print(f"{nom:<34}{attendu[nom]:>8}   {cases[0]:>16}   {cases[1]:>16}")

instance                           attendu             algo 1             algo 2
cli_model_sat.cnf                      SAT      SAT ok  0.13s      SAT ok  0.13s
edge_all_negative_sat.cnf              SAT      SAT ok  0.13s      SAT ok  0.12s
edge_duplicate_literals_unsat.cnf    UNSAT    UNSAT ok  0.07s    UNSAT ok  0.08s
edge_empty_clause_unsat.cnf          UNSAT    UNSAT ok  0.09s    UNSAT ok  0.08s
edge_empty_formula_sat.cnf             SAT      SAT ok  0.08s      SAT ok  0.08s
edge_no_clause_sat.cnf                 SAT      SAT ok  0.08s      SAT ok  0.08s
edge_tautology_sat.cnf                 SAT      SAT ok  0.10s      SAT ok  0.10s
public-01.cnf                          SAT      SAT ok  0.10s      SAT ok  0.10s
public-02.cnf                          SAT      SAT ok  0.09s      SAT ok  0.09s
public-03.cnf                        UNSAT            timeout            timeout
public-04.cnf                          SAT      SAT ok  0.12s      SAT ok  0.12s
public-05.cnf               

**c.** Grilles Meowdoku de la question 1 : DPLL trouve une solution, contrôlée par `verifier_solution`.

In [13]:
for N in (4, 6, 8, 10, 12):
    g = lire_grille(f"grilles/meow_{N:02d}_0.txt")
    nv, cl = grille_vers_cnf(g)
    t0 = time.perf_counter()
    sat, v = dpll(cl, random.Random(0))
    duree = time.perf_counter() - t0
    chats = chats_depuis_valuation({p for p, b in v.items() if b}, g.M)
    print(f"{N:>2}x{N:<2} : SAT = {sat}, règles violées = {verifier_solution(g, chats)}, {duree:.3f} s")

 4x4  : SAT = True, règles violées = [], 0.000 s
 6x6  : SAT = True, règles violées = [], 0.003 s
 8x8  : SAT = True, règles violées = [], 0.008 s
10x10 : SAT = True, règles violées = [], 0.073 s
12x12 : SAT = True, règles violées = [], 0.087 s
